In [0]:
from pyspark.sql import functions as F

fact_sales = spark.table(
    "retail_demo.silver.fact_sales"
)

monthly_category = (
    fact_sales
    .groupBy(
        F.trunc("order_date", "month")
        .alias("month_date"),
        F.col("category")
        .alias("dimension_value")
    )
    .agg(
        F.round(
            F.sum("net_sales"),
            2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"),
            2
        ).alias("gross_profit"),

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers")
    )
    .withColumn(
        "dimension_type",
        F.lit("Category")
    )
)

monthly_channel = (
    fact_sales
    .groupBy(
        F.trunc("order_date", "month")
        .alias("month_date"),
        F.col("sales_channel")
        .alias("dimension_value")
    )
    .agg(
        F.round(
            F.sum("net_sales"),
            2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"),
            2
        ).alias("gross_profit"),

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers")
    )
    .withColumn(
        "dimension_type",
        F.lit("Channel")
    )
)

monthly_region = (
    fact_sales
    .groupBy(
        F.trunc("order_date", "month")
        .alias("month_date"),
        F.col("region")
        .alias("dimension_value")
    )
    .agg(
        F.round(
            F.sum("net_sales"),
            2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"),
            2
        ).alias("gross_profit"),

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers")
    )
    .withColumn(
        "dimension_type",
        F.lit("Region")
    )
)

monthly_sales_drivers = (
    monthly_category
    .unionByName(monthly_channel)
    .unionByName(monthly_region)
    .select(
        "month_date",
        "dimension_type",
        "dimension_value",
        "net_sales",
        "gross_profit",
        "total_orders",
        "unique_customers"
    )
)

(
    monthly_sales_drivers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.monthly_sales_drivers"
    )
)

print(
    "Created retail_demo.gold.monthly_sales_drivers"
)




In [0]:
monthly_kpi_df = spark.sql("""
SELECT
    MAKE_DATE(
        order_year,
        order_month,
        1
    ) AS month_date,

    net_sales,
    gross_profit,
    total_orders,
    unique_customers,
    avg_order_value,
    profit_margin_pct

FROM retail_demo.gold.monthly_sales

ORDER BY month_date
""")

In [0]:
monthly_driver_df = spark.sql("""
SELECT
    month_date,
    dimension_type,
    dimension_value,
    net_sales,
    gross_profit,
    total_orders,
    unique_customers

FROM retail_demo.gold.monthly_sales_drivers

WHERE dimension_value IS NOT NULL

ORDER BY
    month_date,
    dimension_type,
    net_sales DESC
""")

In [0]:
import json

monthly_kpi_pd = monthly_kpi_df.toPandas()

monthly_driver_pd = monthly_driver_df.toPandas()

monthly_kpi_pd["month_date"] = (
    monthly_kpi_pd["month_date"]
    .astype(str)
)

monthly_driver_pd["month_date"] = (
    monthly_driver_pd["month_date"]
    .astype(str)
)

kpi_json = json.dumps(
    monthly_kpi_pd.to_dict(
        orient="records"
    ),
    default=str
)

driver_json = json.dumps(
    monthly_driver_pd.to_dict(
        orient="records"
    ),
    default=str
)

In [0]:
html = """
<div id="salesMomentum">

<style>

#salesMomentum {
    font-family: Arial, sans-serif;
    background: linear-gradient(
        135deg,
        #071426 0%,
        #0b2442 55%,
        #12355d 100%
    );
    color: #ffffff;
    padding: 24px;
    border-radius: 18px;
    min-height: 610px;
    box-sizing: border-box;
}

#salesMomentum .header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    gap: 20px;
    margin-bottom: 22px;
}

#salesMomentum .title {
    font-size: 26px;
    font-weight: 700;
}

#salesMomentum .subtitle {
    margin-top: 6px;
    color: #a9bdd5;
    font-size: 13px;
}

#salesMomentum .monthBadge {
    padding: 10px 16px;
    background: rgba(255, 255, 255, 0.10);
    border: 1px solid rgba(255, 255, 255, 0.16);
    border-radius: 999px;
    font-size: 16px;
    font-weight: 700;
    white-space: nowrap;
}

#salesMomentum .cards {
    display: grid;
    grid-template-columns: repeat(4, minmax(140px, 1fr));
    gap: 12px;
    margin-bottom: 22px;
}

#salesMomentum .card {
    background: rgba(255, 255, 255, 0.075);
    border: 1px solid rgba(255, 255, 255, 0.12);
    border-radius: 13px;
    padding: 15px;
}

#salesMomentum .cardLabel {
    color: #a9bdd5;
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 0.7px;
}

#salesMomentum .cardValue {
    margin-top: 7px;
    font-size: 21px;
    font-weight: 700;
}

#salesMomentum .cardFooter {
    margin-top: 5px;
    font-size: 11px;
    color: #7dd3fc;
}

#salesMomentum .positive {
    color: #59e3ae;
}

#salesMomentum .negative {
    color: #ff8a9a;
}

#salesMomentum .panel {
    background: rgba(255, 255, 255, 0.055);
    border: 1px solid rgba(255, 255, 255, 0.10);
    border-radius: 14px;
    padding: 18px;
}

#salesMomentum .panelHeader {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 14px;
    margin-bottom: 17px;
}

#salesMomentum .panelTitle {
    font-size: 17px;
    font-weight: 700;
}

#salesMomentum .buttons {
    display: flex;
    flex-wrap: wrap;
    gap: 7px;
}

#salesMomentum button {
    border: 1px solid rgba(255, 255, 255, 0.17);
    background: rgba(255, 255, 255, 0.08);
    color: #dbeafe;
    padding: 7px 13px;
    border-radius: 999px;
    cursor: pointer;
    font-size: 12px;
}

#salesMomentum button.active {
    background: #38bdf8;
    color: #06233b;
    border-color: #38bdf8;
    font-weight: 700;
}

#salesMomentum button:focus {
    outline: 2px solid #ffffff;
    outline-offset: 2px;
}

#salesMomentum .bars {
    min-height: 250px;
}

#salesMomentum .barRow {
    display: grid;
    grid-template-columns:
        minmax(110px, 180px)
        1fr
        minmax(95px, 130px);
    align-items: center;
    gap: 12px;
    margin-bottom: 12px;
}

#salesMomentum .rankName {
    overflow: hidden;
    text-overflow: ellipsis;
    white-space: nowrap;
    font-size: 13px;
    font-weight: 600;
}

#salesMomentum .rank {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    width: 22px;
    height: 22px;
    margin-right: 7px;
    border-radius: 7px;
    background: rgba(255, 255, 255, 0.11);
    color: #bae6fd;
    font-size: 11px;
}

#salesMomentum .barTrack {
    height: 25px;
    background: rgba(255, 255, 255, 0.08);
    border-radius: 7px;
    overflow: hidden;
}

#salesMomentum .barFill {
    width: 0%;
    height: 100%;
    border-radius: 7px;
    background: linear-gradient(
        90deg,
        #38bdf8 0%,
        #818cf8 55%,
        #c084fc 100%
    );
    transition: width 900ms cubic-bezier(
        0.22,
        1,
        0.36,
        1
    );
}

#salesMomentum .barValue {
    text-align: right;
    font-size: 12px;
    font-weight: 700;
    color: #dbeafe;
}

#salesMomentum .footer {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 12px;
    margin-top: 16px;
}

#salesMomentum .progressTrack {
    flex: 1;
    height: 5px;
    border-radius: 999px;
    background: rgba(255, 255, 255, 0.10);
    overflow: hidden;
}

#salesMomentum .progressFill {
    height: 100%;
    width: 0%;
    background: #38bdf8;
    transition: width 500ms ease;
}

#salesMomentum .controlButtons {
    display: flex;
    gap: 7px;
}

#salesMomentum .status {
    color: #a9bdd5;
    font-size: 11px;
    min-height: 16px;
}

@media (max-width: 800px) {

    #salesMomentum .cards {
        grid-template-columns:
            repeat(2, minmax(135px, 1fr));
    }

    #salesMomentum .header,
    #salesMomentum .panelHeader,
    #salesMomentum .footer {
        align-items: stretch;
        flex-direction: column;
    }

}

</style>


<div class="header">

    <div>
        <div class="title">
            Retail Sales Momentum
        </div>

        <div class="subtitle">
            Monthly performance and changing business drivers
        </div>
    </div>

    <div
        id="momentumMonth"
        class="monthBadge">
    </div>

</div>


<div class="cards">

    <div class="card">

        <div class="cardLabel">
            Net Sales
        </div>

        <div
            id="momentumSales"
            class="cardValue">
        </div>

        <div
            id="momentumGrowth"
            class="cardFooter">
        </div>

    </div>


    <div class="card">

        <div class="cardLabel">
            Gross Profit
        </div>

        <div
            id="momentumProfit"
            class="cardValue">
        </div>

        <div
            id="momentumMargin"
            class="cardFooter">
        </div>

    </div>


    <div class="card">

        <div class="cardLabel">
            Total Orders
        </div>

        <div
            id="momentumOrders"
            class="cardValue">
        </div>

        <div class="cardFooter">
            Distinct monthly orders
        </div>

    </div>


    <div class="card">

        <div class="cardLabel">
            Unique Customers
        </div>

        <div
            id="momentumCustomers"
            class="cardValue">
        </div>

        <div
            id="momentumAov"
            class="cardFooter">
        </div>

    </div>

</div>


<div class="panel">

    <div class="panelHeader">

        <div>

            <div
                id="driverTitle"
                class="panelTitle">
                Top Categories
            </div>

            <div
                id="statusText"
                class="status"
                aria-live="polite">
            </div>

        </div>


        <div class="buttons">

            <button
                type="button"
                data-type="Category"
                class="dimensionButton active">
                Categories
            </button>

            <button
                type="button"
                data-type="Channel"
                class="dimensionButton">
                Channels
            </button>

            <button
                type="button"
                data-type="Region"
                class="dimensionButton">
                Regions
            </button>

        </div>

    </div>


    <div
        id="momentumBars"
        class="bars">
    </div>

</div>


<div class="footer">

    <div class="progressTrack">

        <div
            id="momentumProgress"
            class="progressFill">
        </div>

    </div>


    <div class="controlButtons">

        <button
            id="playPauseButton"
            type="button">
            Pause
        </button>

        <button
            id="replayButton"
            type="button">
            Replay
        </button>

    </div>

</div>


<script>

(function() {

    const root = document.getElementById(
        "salesMomentum"
    );

    const kpiData = __KPI_DATA__;
    const driverData = __DRIVER_DATA__;

    let monthIndex = 0;
    let selectedType = "Category";
    let timer = null;
    let playing = true;


    const monthElement =
        root.querySelector("#momentumMonth");

    const salesElement =
        root.querySelector("#momentumSales");

    const growthElement =
        root.querySelector("#momentumGrowth");

    const profitElement =
        root.querySelector("#momentumProfit");

    const marginElement =
        root.querySelector("#momentumMargin");

    const ordersElement =
        root.querySelector("#momentumOrders");

    const customersElement =
        root.querySelector("#momentumCustomers");

    const aovElement =
        root.querySelector("#momentumAov");

    const barsElement =
        root.querySelector("#momentumBars");

    const progressElement =
        root.querySelector("#momentumProgress");

    const statusElement =
        root.querySelector("#statusText");

    const titleElement =
        root.querySelector("#driverTitle");

    const playPauseButton =
        root.querySelector("#playPauseButton");

    const replayButton =
        root.querySelector("#replayButton");


    function safeNumber(value) {

        const parsed = Number(value);

        if (!Number.isFinite(parsed)) {
            return 0;
        }

        return parsed;
    }


    function formatMoney(value) {

        return new Intl.NumberFormat(
            "en-IN",
            {
                style: "currency",
                currency: "INR",
                notation: "compact",
                maximumFractionDigits: 1
            }
        ).format(
            safeNumber(value)
        );
    }


    function formatNumber(value) {

        return new Intl.NumberFormat(
            "en-IN"
        ).format(
            safeNumber(value)
        );
    }


    function formatPercent(value) {

        return new Intl.NumberFormat(
            "en-IN",
            {
                maximumFractionDigits: 1
            }
        ).format(
            safeNumber(value)
        ) + "%";
    }


    function formatMonth(dateValue) {

        const date = new Date(
            dateValue + "T00:00:00"
        );

        return date.toLocaleDateString(
            "en-IN",
            {
                month: "long",
                year: "numeric"
            }
        );
    }


    function calculateGrowth(
        currentValue,
        previousValue
    ) {

        const current =
            safeNumber(currentValue);

        const previous =
            safeNumber(previousValue);

        if (previous === 0) {
            return null;
        }

        return (
            (current - previous) /
            previous
        ) * 100;
    }


    function renderBars(monthDate) {

        const rows = driverData
            .filter(function(row) {

                return (
                    row.month_date === monthDate &&
                    row.dimension_type ===
                        selectedType
                );

            })
            .sort(function(left, right) {

                return (
                    safeNumber(right.net_sales) -
                    safeNumber(left.net_sales)
                );

            })
            .slice(0, 5);


        barsElement.innerHTML = "";

        if (selectedType === "Category") {
            titleElement.textContent =
                "Top Categories";
        }

        if (selectedType === "Channel") {
            titleElement.textContent =
                "Top Sales Channels";
        }

        if (selectedType === "Region") {
            titleElement.textContent =
                "Top Regions";
        }


        if (rows.length === 0) {

            statusElement.textContent =
                "No records available for this month.";

            return;
        }


        statusElement.textContent =
            "Ranked by monthly Net Sales";


        const maximum = Math.max(
            1,
            safeNumber(
                rows[0].net_sales
            )
        );


        rows.forEach(
            function(row, index) {

                const width = Math.max(
                    0,
                    Math.min(
                        100,
                        (
                            safeNumber(
                                row.net_sales
                            ) /
                            maximum
                        ) * 100
                    )
                );


                const rowElement =
                    document.createElement("div");

                rowElement.className =
                    "barRow";


                const nameElement =
                    document.createElement("div");

                nameElement.className =
                    "rankName";


                const rankElement =
                    document.createElement("span");

                rankElement.className =
                    "rank";

                rankElement.textContent =
                    String(index + 1);


                const labelElement =
                    document.createElement("span");

                labelElement.textContent =
                    String(
                        row.dimension_value ||
                        "Unknown"
                    );


                nameElement.appendChild(
                    rankElement
                );

                nameElement.appendChild(
                    labelElement
                );


                const trackElement =
                    document.createElement("div");

                trackElement.className =
                    "barTrack";


                const fillElement =
                    document.createElement("div");

                fillElement.className =
                    "barFill";


                trackElement.appendChild(
                    fillElement
                );


                const valueElement =
                    document.createElement("div");

                valueElement.className =
                    "barValue";

                valueElement.textContent =
                    formatMoney(
                        row.net_sales
                    );


                rowElement.appendChild(
                    nameElement
                );

                rowElement.appendChild(
                    trackElement
                );

                rowElement.appendChild(
                    valueElement
                );


                barsElement.appendChild(
                    rowElement
                );


                window.setTimeout(
                    function() {

                        fillElement.style.width =
                            width + "%";

                    },
                    60 + index * 75
                );

            }
        );
    }


    function renderMonth() {

        if (kpiData.length === 0) {

            statusElement.textContent =
                "No monthly KPI records available.";

            playing = false;

            clearInterval(timer);

            return;
        }


        const row =
            kpiData[monthIndex];


        const previousRow =
            monthIndex > 0
                ? kpiData[monthIndex - 1]
                : null;


        monthElement.textContent =
            formatMonth(
                row.month_date
            );


        salesElement.textContent =
            formatMoney(
                row.net_sales
            );


        profitElement.textContent =
            formatMoney(
                row.gross_profit
            );


        ordersElement.textContent =
            formatNumber(
                row.total_orders
            );


        customersElement.textContent =
            formatNumber(
                row.unique_customers
            );


        marginElement.textContent =
            formatPercent(
                row.profit_margin_pct
            ) +
            " profit margin";


        aovElement.textContent =
            formatMoney(
                row.avg_order_value
            ) +
            " average order value";


        const growth =
            previousRow
                ? calculateGrowth(
                    row.net_sales,
                    previousRow.net_sales
                )
                : null;


        growthElement.classList.remove(
            "positive",
            "negative"
        );


        if (growth === null) {

            growthElement.textContent =
                "Starting month";

        } else {

            const prefix =
                growth >= 0
                    ? "+"
                    : "";

            growthElement.textContent =
                prefix +
                formatPercent(growth) +
                " vs previous month";

            growthElement.classList.add(
                growth >= 0
                    ? "positive"
                    : "negative"
            );
        }


        renderBars(
            row.month_date
        );


        const progress =
            (
                (monthIndex + 1) /
                kpiData.length
            ) * 100;


        progressElement.style.width =
            progress + "%";
    }


    function nextMonth() {

        if (!playing) {
            return;
        }


        monthIndex++;


        if (
            monthIndex >=
            kpiData.length
        ) {

            monthIndex =
                kpiData.length - 1;

            playing = false;

            playPauseButton.textContent =
                "Play";

            clearInterval(timer);

            return;
        }


        renderMonth();
    }


    function startTimer() {

        clearInterval(timer);

        timer = window.setInterval(
            nextMonth,
            2200
        );
    }


    function replay() {

        monthIndex = 0;
        playing = true;

        playPauseButton.textContent =
            "Pause";

        renderMonth();
        startTimer();
    }


    root.querySelectorAll(
        ".dimensionButton"
    ).forEach(
        function(button) {

            button.addEventListener(
                "click",
                function() {

                    selectedType =
                        button.dataset.type;


                    root.querySelectorAll(
                        ".dimensionButton"
                    ).forEach(
                        function(item) {

                            item.classList.remove(
                                "active"
                            );

                        }
                    );


                    button.classList.add(
                        "active"
                    );


                    if (kpiData.length > 0) {

                        renderBars(
                            kpiData[
                                monthIndex
                            ].month_date
                        );
                    }

                }
            );

        }
    );


    playPauseButton.addEventListener(
        "click",
        function() {

            playing = !playing;

            playPauseButton.textContent =
                playing
                    ? "Pause"
                    : "Play";


            if (playing) {

                if (
                    monthIndex >=
                    kpiData.length - 1
                ) {

                    monthIndex = 0;
                    renderMonth();
                }

                startTimer();

            } else {

                clearInterval(timer);
            }

        }
    );


    replayButton.addEventListener(
        "click",
        replay
    );


    if (kpiData.length > 0) {

        renderMonth();
        startTimer();

    } else {

        statusElement.textContent =
            "No monthly KPI data available.";

        playPauseButton.disabled = true;
        replayButton.disabled = true;
    }

})();

</script>

</div>
"""


html = (
    html
    .replace(
        "__KPI_DATA__",
        kpi_json
    )
    .replace(
        "__DRIVER_DATA__",
        driver_json
    )
)


displayHTML(html)